# MenuScan TN · 03.04 Publication des résultats dans le schéma `ml` (T7.4)

Autrice : Sarra Abdeljaouad

**Objet.** Copier les résultats de la phase 7 dans le schéma `ml` de l'entrepôt, pour Power BI, puis les contrôler.

**À lancer en dernier**, après tous les autres notebooks de la phase 7.

**Garanties.**
- **Une seule transaction** : en cas d'erreur, rien n'est modifié, et Power BI garde la publication précédente.
- Le script `00_commun/sql/01_schema_ml.sql` crée le schéma s'il n'existe pas. Il peut être relancé sans risque.
- Chaque publication est tracée dans `ml.journal_publication`.

In [1]:
import sys
sys.path.insert(0, "../00_commun")

import json

import pandas as pd

from menuscan_analytics import config, publication

config.style()
T = config.DOSSIER_TABLES
lire_csv = lambda nom: pd.read_csv(T / f"{nom}.csv", sep=";", encoding="utf-8-sig")
fiche = json.loads((config.DOSSIER_MODELES / "fiches" / "resiliation.json").read_text(encoding="utf-8"))

## 1. Préparation des tables

In [2]:
scores = lire_csv("ML02_scores_2026_12")
score_resiliation = pd.DataFrame({
    "etablissement_id": scores.etablissement_id, "mois_observe": pd.to_datetime(scores.mois).dt.date,
    "probabilite": scores.probabilite.round(5), "rang": scores.rang, "alerte": scores.alerte.astype(bool),
    "facteur_1": scores.facteur_1, "facteur_2": scores.facteur_2, "facteur_3": scores.facteur_3,
    "version_modele": "resiliation v1 (27/11/2026)"})

cand = lire_csv("ML03_candidats_commande")
candidat_commande = cand[["etablissement_id", "sessions_table_jour_3m", "seuil_usage", "probabilite_resiliation"]].round(5)

colonnes_regles = ["antecedent", "consequent", "commandes_ensemble", "confiance", "lift_ajuste", "lift_2026"]
regle_association = pd.concat([lire_csv("DM01_regles_categories")[colonnes_regles].assign(niveau="categorie"),
                               lire_csv("DM01_regles_plats")[colonnes_regles].assign(niveau="plat")])
regle_association["commandes_ensemble"] = regle_association.commandes_ensemble.astype(int)
regle_association = regle_association.round({"confiance": 5, "lift_ajuste": 4, "lift_2026": 4})

recommandation_plat = lire_csv("DM01_recommandations_pro")

elasticite_categorie = pd.concat([lire_csv("DM02_elasticites_categories"), lire_csv("DM02_elasticite_globale")])
elasticite_categorie = elasticite_categorie.rename(columns={"effet_hausse_5pc_sur_CA": "effet_hausse_5pc_sur_ca"}).round({"elasticite": 4, "bas_95": 4, "haut_95": 4,
                                                      "effet_hausse_5pc_sur_ca": 5})

registre_modele = lire_csv("ML03_registre_modeles").round({"valeur": 5, "bas_95": 5, "haut_95": 5})

tables = {"score_resiliation": score_resiliation, "candidat_commande": candidat_commande,
          "regle_association": regle_association, "recommandation_plat": recommandation_plat,
          "elasticite_categorie": elasticite_categorie, "registre_modele": registre_modele}
pd.Series({n: len(t) for n, t in tables.items()}).to_frame("lignes à publier")

,lignes à publier
score_resiliation,265
candidat_commande,47
regle_association,13
recommandation_plat,749
elasticite_categorie,21
registre_modele,11


## 2. Publication

In [3]:
journal = publication.publier(tables)
journal

,publie_le,nom_table,lignes,statut,message
0,2026-11-28 15:21:48,score_resiliation,265,OK,None
1,2026-11-28 15:21:48,candidat_commande,47,OK,None
2,2026-11-28 15:21:48,regle_association,13,OK,None
3,2026-11-28 15:21:48,recommandation_plat,749,OK,None
4,2026-11-28 15:21:48,elasticite_categorie,21,OK,None
5,2026-11-28 15:21:48,registre_modele,11,OK,None


## 3. Contrôles après publication (chaque valeur doit être 0)

In [4]:
controles = publication.controler()
config.enregistrer_table(controles, "ML04_controles_publication", index=False)
assert (controles.statut == "OK").all(), controles[controles.statut != "OK"]
print(f"{len(controles)} contrôles sur {len(controles)} OK")
controles

8 contrôles sur 8 OK


,code,controle,valeur,statut
0,P1,Scores : un par établissement actif au 31/12/2026,0,OK
1,P2,Scores : établissements inconnus de dim_etabli...,0,OK
2,P3,Scores : alertes = 10 % arrondi au supérieur,0,OK
3,P4,Recommandations : établissements hors formule ...,0,OK
4,P5,Recommandations : plat recommandé absent de la...,0,OK
5,P6,Candidats : établissements qui ne sont pas en ...,0,OK
6,P7,Élasticités : intervalle qui ne contient pas l...,0,OK
7,P8,Journal : tables publiées lors de la dernière ...,0,OK


**Suite.** Dans Power BI, cliquer sur **Actualiser**. Les pages TB-4 (Pilotage ML) et TB-8 (Prix) lisent maintenant le schéma `ml`. La page Recette vérifie les nouvelles valeurs (lignes R42 et suivantes).